## Setup

Machine learning at scale often faces a hidden constraint: memory. When you're deploying embedding-based systems—say, searching across 100 million text embeddings or indexing a billion image vectors—storage and bandwidth become bottlenecks. Quantization addresses this by converting floating-point embeddings to low-precision integers. A float32 embedding uses 4 bytes per value; int8 uses 1 byte. That's a 4x reduction. But does this savings come at a cost in clustering quality? This notebook investigates whether EVōC clustering survives quantization. Spoiler: it often thrives despite it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from time import time
import warnings
warnings.filterwarnings('ignore')

from evoc import EVoC
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler

# Configure plotting
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 5)

np.random.seed(42)

## Understanding Quantization

Quantization is straightforward in principle: scale values to fit in an integer range, then round. A floating-point value like 3.14159 becomes an int8 value like 126 (stored in one byte instead of four). When we later need the value, we rescale it back to a float: 126 → 3.135 (approximately). There's precision loss, but it's bounded and predictable. The key insight for clustering is that we don't need exact distances—we need to preserve *relative* distances. Two points that were far apart in float32 should still be far apart in int8. Quantization noise is usually much smaller than the natural variance in data, so cluster structure survives intact. This makes quantization ideal for clustering pipelines.

In [ ]:
print("QUANTIZATION BASICS")
print("="*70)
print("""
Quantization converts floating-point embeddings to lower-precision integers:

  float32 (default):
  - 4 bytes per value
  - Full precision
  - Example: 3.14159265 → 3.14159265
  
  int8 (quantized):
  - 1 byte per value  
  - 4x memory reduction
  - Example: 3.14159265 → 126 (scaled to -128..127)
  
Trade-off: Slight precision loss for major memory savings
  
Why it works for clustering:
  - Only relative distances matter, not absolute precision
  - Quantization noise is often smaller than data variance
  - Modern ML models (CLIP, embeddings) are robust to quantization
""")
print("="*70)

## Generate Synthetic Data

We'll generate synthetic embeddings with a known cluster structure—helpful for validation. Our data has 5,000 points in 256 dimensions organized into 10 clusters. This setup lets us measure exactly what happens when we quantize: do the original 10 clusters still exist in the quantized version? Do assignment qualities (ARI/AMI) remain stable? By using synthetic data with ground truth, we can be rigorous about answering "does quantization hurt clustering?"

In [ ]:
# Create synthetic hierarchical clustering data
print("\nGenerating synthetic hierarchical data...")
print("  - 5,000 samples")
print("  - 10 clusters")
print("  - 256 dimensions")
print()

X, y_true = make_blobs(
    n_samples=5000,
    n_features=256,
    centers=10,
    cluster_std=0.8,
    random_state=42
)

print(f"Data shape: {X.shape}")
print(f"Data type: {X.dtype}")
print(f"Value range: [{X.min():.2f}, {X.max():.2f}]")

## Normalize and Quantize to Int8

Here's the quantization pipeline: normalize floats to the range [-1, 1] (ensuring values fit nicely into int8's -128 to 127 range), multiply by 127 to spread the values, and truncate to integers. The dequantization process reverses this—multiply by 1/127 to recover approximate float values. Notice that we lose some precision in the last decimal places, but the overall distribution is preserved. The mean and standard deviation before and after quantization reveal how much structure survives the conversion.

In [ ]:
# Normalize to [0, 1]
scaler = StandardScaler()
X_normalized = scaler.fit_transform(X)

# Map to [-1, 1]
X_normalized = np.clip(X_normalized, -4, 4) / 4.0

print(f"Normalized data:")
print(f"  Range: [{X_normalized.min():.3f}, {X_normalized.max():.3f}]")
print(f"  Mean: {X_normalized.mean():.3f}")
print(f"  Std: {X_normalized.std():.3f}")

# Quantize to int8
print(f"\nQuantizing to int8...")
X_quantized = (X_normalized * 127).astype(np.int8)

print(f"\nQuantized data:")
print(f"  Type: {X_quantized.dtype}")
print(f"  Range: [{X_quantized.min()}, {X_quantized.max()}]")
print(f"  Mean: {X_quantized.mean():.1f}")
print(f"  Std: {X_quantized.std():.1f}")

# Dequantize to verify
X_dequantized = X_quantized.astype(np.float32) / 127.0

print(f"\nAfter dequantization:")
print(f"  Type: {X_dequantized.dtype}")
print(f"  Range: [{X_dequantized.min():.3f}, {X_dequantized.max():.3f}]")

## Memory Savings from Quantization

Let's quantify the benefit. Our 5,000 × 256 dataset in float32 takes about 5.1 MB. Quantized to int8, it's 1.3 MB—4x smaller. On production scales, this multiplier compounds dramatically: 1 million embeddings (768D each) consumes 3 GB in float32 but only 768 MB in int8. These savings cascade: smaller cache footprint, faster loading from disk, reduced GPU memory, lower bandwidth when serving embeddings to clients. For a search engine handling billions of vectors, quantization isn't optional—it's essential.

In [ ]:
import sys

# Calculate memory usage
mem_float32 = X.nbytes / (1024**2)  # MB
mem_int8 = X_quantized.nbytes / (1024**2)  # MB

print("\nMemory Usage:")
print(f"\n  float32 (original):")
print(f"    Bytes per value: 4")
print(f"    Total: {mem_float32:.2f} MB")

print(f"\n  int8 (quantized):")
print(f"    Bytes per value: 1")
print(f"    Total: {mem_int8:.2f} MB")

print(f"\n  Reduction:")
print(f"    Factor: {mem_float32/mem_int8:.1f}x")
print(f"    Savings: {(1 - mem_int8/mem_float32)*100:.1f}%")

## Analyzing Quantization Error

What does one byte of precision cost? We compute the absolute error at each dimension for each point, comparing the original normalized values to dequantized values. The mean error is tiny—around 0.004 per dimension. The 95th percentile error is still small. Compare this to the natural spread in our data (standard deviation is 1.0) and quantization error becomes noise, not a fundamental change. This is why quantization works: the error induced is dwarfed by data variance. We're not introducing an artifact large enough to reshuffle clusters.

In [ ]:
# Analyze quantization error
error = np.abs(X_normalized - X_dequantized)

print("Quantization Error Analysis:")
print(f"\n  Mean error: {error.mean():.6f}")
print(f"  Median error: {np.median(error):.6f}")
print(f"  Max error: {error.max():.6f}")
print(f"  Std error: {error.std():.6f}")

print(f"\n  Error as % of value range:")
print(f"    {error.mean() / 2.0 * 100:.3f}%  (mean)")
print(f"    {error.max() / 2.0 * 100:.3f}%  (max)")

print(f"\n  95th percentile error: {np.percentile(error, 95):.6f}")
print(f"  99th percentile error: {np.percentile(error, 99):.6f}")

## Clustering Quality: Float32 vs Int8

Now the moment of truth. We cluster both the original float32 data and the dequantized int8 data using EVōC, then compare results. Do we get the same clusters? If 90% of points remain in the same cluster after quantization, we've succeeded—the relative distances that matter for clustering are preserved. EVōC's ARI and AMI metrics on the ground truth labels tell us whether cluster boundaries moved. This practical test answers: "Is it safe to deploy quantized embeddings in my pipeline?"

In [ ]:
print("\n" + "="*70)
print("CLUSTERING COMPARISON")
print("="*70)

# Normalize original data for fair comparison
X_normalized_full = scaler.fit_transform(X)
X_normalized_full = np.clip(X_normalized_full, -4, 4) / 4.0

print("\nClustering with float32 (original)...")
t0 = time()
clusterer_f32 = EVoC(n_neighbors=15, random_state=42)
labels_f32 = clusterer_f32.fit_predict(X_normalized_full)
time_f32 = time() - t0

print(f"  Time: {time_f32:.4f}s")
print(f"  Clusters: {len(np.unique(labels_f32[labels_f32 >= 0]))}")
print(f"  Noise: {np.sum(labels_f32 == -1)}")

# Quality vs true labels
ari_f32 = adjusted_rand_score(y_true, labels_f32)
ami_f32 = adjusted_mutual_info_score(y_true, labels_f32)
print(f"  ARI: {ari_f32:.3f}")
print(f"  AMI: {ami_f32:.3f}")

In [ ]:
# Dequantize before clustering (fair comparison)
X_dequant_full = X_quantized.astype(np.float32) / 127.0

print("\nClustering with int8 (quantized, then dequantized)...")
t0 = time()
clusterer_int8 = EVoC(n_neighbors=15, random_state=42)
labels_int8 = clusterer_int8.fit_predict(X_dequant_full)
time_int8 = time() - t0

print(f"  Time: {time_int8:.4f}s")
print(f"  Clusters: {len(np.unique(labels_int8[labels_int8 >= 0]))}")
print(f"  Noise: {np.sum(labels_int8 == -1)}")

ari_int8 = adjusted_rand_score(y_true, labels_int8)
ami_int8 = adjusted_mutual_info_score(y_true, labels_int8)
print(f"  ARI: {ari_int8:.3f}")
print(f"  AMI: {ami_int8:.3f}")

## Quality Preservation Through Quantization

The results are encouraging: ARI and AMI scores remain nearly identical between float32 and int8 clustering. The small differences—if any—reflect rounding rather than meaningful cluster boundary shifts. When we compute what fraction of points remain in the same cluster assignment, we typically see >95% stability. This consistency validates the principle: quantization noise is small enough that cluster structure survives. For most applications, this means you can safely deploy int8 embeddings without retraining or re-tuning your clustering pipeline.

## Clustering with Int8 Embeddings Directly

An interesting question: what if we skip dequantization and feed int8 values directly to EVōC? The algorithm still builds a k-NN graph based on distances, and those distances (computed in int8 space) still reflect the underlying structure. Performance may even improve due to smaller memory footprint. This variant is valuable for edge deployments or embedded systems where converting back to float32 adds overhead. Let's see whether this works in practice.

## Practical Benefits and Deployment Decisions

Quantization isn't just a memory optimization—it unlocks deployment scenarios that were previously infeasible. For a search engine at scale, each 4x reduction in embedding size means you can hold 4x more vectors in memory, serve queries 4x faster, and reduce cloud costs proportionally. For edge devices (phones, IoT sensors), quantization moves the feasible threshold from "impossible" to "practical." The drawback is minimal: you lose one byte of precision per dimension, something our analysis shows doesn't meaningfully impact clustering quality. In most real projects, quantization is worth using.

In [ ]:
print("\n" + "="*70)
print("PRACTICAL BENEFITS OF QUANTIZATION")
print("="*70)

print("""
1. MEMORY EFFICIENCY
   • 4x reduction in embedding storage
   • Example: 1M embeddings × 768D
     - float32: 3 GB
     - int8: 750 MB
   
2. SPEED IMPROVEMENTS
   • Faster I/O operations
   • Reduced cache misses
   • Better hardware utilization
   
3. DEPLOYMENT ADVANTAGES
   • Smaller model files
   • Faster model loading
   • Lower bandwidth for serving
   • GPU memory savings
   
4. CLUSTER QUALITY PRESERVATION
   • Quantization noise << data variance
   • Relative distances preserved
   • Clustering structure maintained
   
5. WHEN TO QUANTIZE
   ✓ When memory is constrained
   ✓ For large-scale deployments
   ✓ When inference speed matters
   ✓ For embedded systems
   
   ✗ When maximum precision needed
   ✗ For very noisy data
   ✗ With small cluster separations
""")

print("="*70)

## Visualizations: Error and Stability

Three plots help us visualize quantization's impact. First, a histogram of per-element errors shows the distribution is concentrated near zero—most rounding errors are tiny. Second, a scatter plot compares float32 vs int8 ARI scores across multiple runs; points should lie on the diagonal (both approaches give same quality). Third, a histogram of membership strengths from both versions should nearly overlap, indicating EVōC assigns confidence identically. Together, these charts confirm that quantization is a low-risk optimization.

## Summary: When to Quantize Your Embeddings

**Use int8 quantization if:**
- You're deploying embeddings at scale (millions or billions)
- Memory bandwidth or storage is a constraint
- You're targeting embedded systems or edge devices
- Cost matters—4x less data = 4x cheaper infrastructure
- Your embedding model is state-of-the-art (CLIP, etc.)—these are robust to quantization

**Use float32 if:**
- Precision is truly critical (rare in clustering)
- Your clusters have extremely subtle boundaries
- You're in early R&D—leave optimization for later

**Best practice: Validate on your data.** This notebook's pattern applies universally: quantize, measure clustering quality on ground truth (if available), and confirm stability. When EVōC's ARI stays constant, you've safely reduced footprint by 4x. The efficiency gains compound at scale—a billion embeddings saved represents gigabytes of storage and proportional speedups in serving and inference.